# Portfolio Analysis Project Assignment


In [1]:
import yfinance as yf
import pandas as pd

In [2]:
selected_seven = ["META", "GOOGL", "DIS", "BBY", "AAL","ZYME", "WRB"]

In [3]:
stocks = yf.download(selected_seven, start="2024-03-24", end="2026-03-24")


[*********************100%***********************]  7 of 7 completed


In [4]:
stocks.head()

Price       Close                                                            \
Ticker        AAL        BBY         DIS       GOOGL        META        WRB   
Date                                                                          
2024-03-25  14.92  74.315781  116.993874  148.839844  499.632843  54.953777   
2024-03-26  14.92  73.708122  117.552574  149.434906  492.550934  54.839714   
2024-03-27  15.30  75.374535  118.581757  149.633270  490.534546  55.872627   
2024-03-28  15.35  75.521828  119.934395  149.692780  482.310272  56.043728   
2024-04-01  15.41  75.162773  119.120850  154.215424  488.041412  55.441723   

Price               High                         ...        Open             \
Ticker       ZYME    AAL        BBY         DIS  ...        META        WRB   
Date                                             ...                          
2024-03-25  10.40  14.93  75.844075  117.042885  ...  502.384210  54.725650   
2024-03-26  10.47  15.12  75.300869  118.013258  ...  501.728706  54.763673   
2024-03-27  10.30  15.30  76.129477  119.610937  ...  495.937918  54.998134   
2024-03-28  10.52  15.50  76.378043  121.287036  ...  489.521396  56.151457   
2024-04-01   9.88  15.61  76.332017  119.973606  ...  483.919363  55.955014   

Price                Volume                                                  \
Ticker       ZYME       AAL      BBY       DIS     GOOGL      META      WRB   
Date                                                                          
2024-03-25  10.51  21392400  2813700  12103600  19229300   8380600  1380450   
2024-03-26  10.48  20898500  2106700  11870000  22149100  11205400  1833300   
2024-03-27  10.51  24789300  3906800  10782800  22879200   9989700  1564500   
2024-03-28  10.35  36396000  2571300  15367400  24485400  15212800  2471550   
2024-04-01  10.47  23249600  1888100   8419700  31730800   9247000  1722750   

Price               
Ticker        ZYME  
Date                
2024-03-25  292600  
2024-03-26  345300  
2024-03-27  398700  
2024-03-28  468400  
2024-04-01  744300  

[5 rows x 35 columns]

In [14]:
price_field="Close"

stock_2y = None
if isinstance(stocks.columns, pd.MultiIndex):
    # yfinance returns multiindex columns if multiple tickers
    if price_field not in stocks.columns.levels[0]:
        raise ValueError(f"Price field '{price_field}' not found. Available: {stocks.columns.levels[0]}")
    stock_2y = stocks[price_field].copy()
else:
    # single ticker case
    stock_2y = stocks[[price_field]].copy()
    stock_2y.columns = selected_seven

stock_2y = stock_2y.dropna(how="all").dropna(axis=1)  # drop any tickers that failed
if stock_2y.shape[1] != len(selected_seven):
    missing = set(selected_seven) - set(stock_2y.columns)
    raise ValueError(f"Missing tickers data: {missing}")

In [15]:
stock_2y.head()

Ticker,AAL,BBY,DIS,GOOGL,META,WRB,ZYME
Date,,,,,,,
2024-03-25,14.92,74.315781,116.993874,148.839844,499.632843,54.953777,10.40
2024-03-26,14.92,73.708122,117.552574,149.434906,492.550934,54.839714,10.47
2024-03-27,15.30,75.374535,118.581757,149.633270,490.534546,55.872627,10.30
2024-03-28,15.35,75.521828,119.934395,149.692780,482.310272,56.043728,10.52
2024-04-01,15.41,75.162773,119.120850,154.215424,488.041412,55.441723,9.88


In [9]:
print(stock_2y.columns)

Index(['AAL', 'BBY', 'DIS', 'GOOGL', 'META', 'WRB', 'ZYME'], dtype='object', name='Ticker')


In [18]:


stock_vs_market = pd.DataFrame({
    "ticker": selected_seven,
    "portfolio_weight": [1/len(selected_seven)] * len(selected_seven),
    "annualized_volatility": stock_2y.tail(63).pct_change().std() * (252**0.5)

})



In [19]:
stock_vs_market

,ticker,portfolio_weight,annualized_volatility
Ticker,,,
AAL,META,0.142857,0.480299
BBY,GOOGL,0.142857,0.354384
DIS,DIS,0.142857,0.282701
GOOGL,BBY,0.142857,0.208616
META,AAL,0.142857,0.348677
WRB,ZYME,0.142857,0.205110
ZYME,WRB,0.142857,0.424655


In [20]:
# SPY, IWM, DIA data pull
market_index = yf.download(["SPY", "IWM", "DIA"], start="2024-03-24", end="2026-03-24")["Close"].dropna()
market_index.head()

[*********************100%***********************]  3 of 3 completed


Ticker,DIA,IWM,SPY
Date,,,
2024-03-25,380.732147,201.043533,507.420959
2024-03-26,380.635162,200.681641,506.483765
2024-03-27,385.198669,205.053879,510.740204
2024-03-28,385.392487,205.699417,510.642456
2024-04-01,383.038055,203.762741,509.754120


In [23]:
# beta calculation with beta of stock_2y vs SPY, IWM, DIA
stock_returns = stock_2y.pct_change()
for index in ["SPY", "IWM", "DIA"]:
    market_ret = market_index[index].pct_change()
    
    stock_vs_market[f"beta_{index}"] = stock_returns.apply(
        lambda x: x.cov(market_ret) / market_ret.var()
    )

In [24]:
stock_vs_market

,ticker,portfolio_weight,annualized_volatility,beta_SPY,beta_IWM,beta_DIA
Ticker,,,,,,
AAL,META,0.142857,0.480299,1.644988,1.337598,1.914545
BBY,GOOGL,0.142857,0.354384,1.150062,0.949296,1.370167
DIS,DIS,0.142857,0.282701,0.910847,0.632253,1.081758
GOOGL,BBY,0.142857,0.208616,1.063886,0.602526,0.874853
META,AAL,0.142857,0.348677,1.363989,0.712716,1.165079
WRB,ZYME,0.142857,0.205110,0.217806,0.185228,0.487813
ZYME,WRB,0.142857,0.424655,0.839614,0.773638,0.920529
